In [1]:
!pip install transformers torch accelerate

In [3]:
!pip install transformers torch accelerate

import warnings, logging, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Suppress warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ✅ Use smaller model (IMPORTANT)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Force CPU (safe)
device = torch.device("cpu")
torch.set_num_threads(1)

print("=" * 60)
print(f"  Model  : {MODEL_NAME}")
print(f"  Device : {device}")
print("=" * 60)
print("Loading model and tokenizer...\n")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model safely
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

model.to(device)
model.eval()

print("Model loaded successfully!\n")

# -------------------------------
# Response Generation Function
# -------------------------------
def generate_response(user_input: str, history: list) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant. "
                "Answer clearly in 2-3 sentences."
            )
        }
    ] + history + [{"role": "user", "content": user_input}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,              # ✅ reduced for speed
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return response if response else "I'm not sure. Please try again."

# -------------------------------
# Chat Loop
# -------------------------------
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant.")
    print("-" * 50)
    print("Type 'exit' or 'quit' to stop.\n")

    history = []

    while True:
        try:
            user_input = input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nChatbot: Session ended.")
            break

        if not user_input:
            continue

        if user_input.lower() in {"exit", "quit"}:
            print("Chatbot: Goodbye!")
            break

        response = generate_response(user_input, history)

        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": response})

        # ✅ Limit history (prevents slowdown)
        if len(history) > 10:
            history = history[-10:]

        print(f"Chatbot: {response}\n")

# Run chatbot
if __name__ == "__main__":
    run_chatbot()

  Model  : Qwen/Qwen2.5-0.5B-Instruct
  Device : cpu
Loading model and tokenizer...




config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Chatbot: Hello! How can I help you today? Is there something specific you'd like to talk about or ask me? I'm here to assist with any questions you might have, whether it's information, advice, or just general conversation. Feel free to reach out anytime!



You:  what is ai?


Chatbot: AI stands for "Artificial Intelligence," which refers to the simulation of human intelligence in machines that are designed to think and learn like humans. AI involves developing algorithms, systems, and software that can process and analyze large amounts of data automatically, allowing them to perform tasks that would typically require human intelligence.

Some key aspects of AI include:

1. Machine learning: Algorithms trained on data can learn from examples



You:  exit


Chatbot: Goodbye!
